# NCCL 点对点通信（P2P）

本文按 [NCCL Point-to-point communication](https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/usage/p2p.html)（NCCL 2.30）整理。示例为学习用简化 C++；实际使用前需完成 communicator、CUDA stream 和错误检查的初始化。


## 1. 基础：rank、peer 与配对规则

自 NCCL 2.7 起，点对点通信可表达任意 rank 间通信图。`peer` 是当前 rank 的通信对端，即另一个 rank 的编号。一次双边数据传输必须由发送端的 `ncclSend(sendbuff, count, datatype, peer, comm, stream)` 与接收端的 `ncclRecv(recvbuff, count, datatype, peer, comm, stream)` 配对；这对 send/recv 的 `count` 和 `datatype` 必须一致。不同 rank 可以使用不同大小、类型和缓冲区，只要每一对实际匹配的操作满足该规则。


## 2. Group：并发推进与死锁

把面向不同 peer、需要并发推进的操作放进同一 `ncclGroupStart()` / `ncclGroupEnd()`。NCCL 在 `GroupEnd` 时一起提交这批操作，而不是让一条收发单独阻塞整个通信图。

这能避免循环等待：例如两个 rank 都先 Send、再 Recv，单独执行时双方发送可能都等待对方的接收；将这两项操作纳入一个 group 后，两个方向可同时推进。

> 同一 group 内，面向不同 peer 的 P2P 操作可以独立推进；面向同一 peer 的多条 P2P 操作则按代码顺序执行。因此对端也必须按相同顺序匹配消息。


## 3. 双边模式：Sendrecv

两个 rank 双向交换数据：双方均发送自己的数据，并接收对方的数据。

```cpp
ncclGroupStart();
ncclSend(sendbuff, sendcount, sendtype, peer, comm, stream);
ncclRecv(recvbuff, recvcount, recvtype, peer, comm, stream);
ncclGroupEnd();
```

当前 rank 的 send 与 peer 上的 recv 必须匹配；反方向同理。`sendcount/sendtype` 与本 rank 的 `recvcount/recvtype` 不必相同。


## 4. 双边模式：Scatter 与 Gather

**Scatter（一对多）**：根 rank 向每个 rank 发送不同数据块；非根 rank 从根接收。根端多次 send 应分组：

```cpp
if (rank == root) {
  ncclGroupStart();
  for (int r = 0; r < nranks; ++r)
    ncclSend(sendbuf[r], count, datatype, r, comm, stream);
  ncclGroupEnd();
} else {
  ncclRecv(recvbuf, count, datatype, root, comm, stream);
}
```

**Gather（多对一）**：是 Scatter 的反向。非根 rank 向根发送，根 rank 在一个 group 内从每个来源接收：

```cpp
if (rank == root) {
  ncclGroupStart();
  for (int r = 0; r < nranks; ++r)
    ncclRecv(recvbuf[r], count, datatype, r, comm, stream);
  ncclGroupEnd();
} else {
  ncclSend(sendbuf, count, datatype, root, comm, stream);
}
```


## 5. 双边模式：All-to-all 与邻居交换

**All-to-all** 中每个 rank 均向所有 peer 发送一块数据，并从所有 peer 接收一块数据：

```cpp
ncclGroupStart();
for (int peer = 0; peer < nranks; ++peer) {
  ncclSend(sendbuf[peer], count, datatype, peer, comm, stream);
  ncclRecv(recvbuf[peer], count, datatype, peer, comm, stream);
}
ncclGroupEnd();
```

**邻居交换** 常用于环或网格中的 halo 交换。环形拓扑的每个 rank 可向右邻居发边界数据、从左邻居接收：

```cpp
int prev = (rank - 1 + nranks) % nranks;
int next = (rank + 1) % nranks;
ncclGroupStart();
ncclSend(right_halo, halo_count, datatype, next, comm, stream);
ncclRecv(left_halo,  halo_count, datatype, prev, comm, stream);
ncclGroupEnd();
```

N 维网格只需扩展为各维度邻居；边界 rank 跳过不存在的邻居。


## 6. 单边通信（RMA）：window、PutSignal 与 WaitSignal

单边通信中，发送方用 `ncclPutSignal()` 将本地数据直接写入 peer 已注册的 window，并同时发出完成信号；接收方用 `ncclWaitSignal()` 等待该信号。只需同步通知时可用 `ncclSignal()`。

RMA 前，需要分配并注册可远程访问的 GPU 缓冲区。文档的对称窗口做法是每个 rank 以相同方式注册：

```cpp
void *sendbuff, *recvbuff;
ncclMemAlloc(&sendbuff, size);
ncclMemAlloc(&recvbuff, size);
ncclWindow_t sendWindow, recvWindow;
ncclCommWindowRegister(comm, sendbuff, size, &sendWindow, NCCL_WIN_COLL_SYMMETRIC);
ncclCommWindowRegister(comm, recvbuff, size, &recvWindow, NCCL_WIN_COLL_SYMMETRIC);
```

完成 stream 中的通信后，先同步，再注销 window 和释放内存。


## 7. PutSignal + WaitSignal：乒乓模式

`ncclWaitSignalDesc_t` 指定要等待的信号：`peer` 是信号来源，`sigIdx` 是信号槽，`opCnt` 是期待的操作计数。一个典型乒乓序列为：rank 1 先 `PutSignal` 写 rank 0 的接收 window；rank 0 的 `WaitSignal` 满足后再 `PutSignal` 写 rank 1；最后 rank 1 的 `WaitSignal` 满足。

```cpp
if (rank == 0) {
  ncclWaitSignal(1, &waitDesc, comm, stream);
  ncclPutSignal(/* 本地源、peer、远端 window、偏移和信号参数 */);
} else {
  ncclPutSignal(/* 本地源、peer、远端 window、偏移和信号参数 */);
  ncclWaitSignal(1, &waitDesc, comm, stream);
}
```

双方都先等待会死锁，因为没有一端会发出第一个信号。


## 8. Signal + WaitSignal：Barrier

Barrier 要求每个 rank 先通知所有 rank“我已经到达”，再等待来自所有 rank 的通知。包括向自己发送 signal，可令逻辑统一。

```cpp
for (int r = 0; r < nranks; ++r)
  waitDescs[r] = {.opCnt = 1, .peer = r, .sigIdx = 0, .ctx = 0};

ncclGroupStart();
for (int r = 0; r < nranks; ++r)
  ncclSignal(r, 0, 0, 0, comm, stream);
ncclGroupEnd();

ncclWaitSignal(nranks, waitDescs, comm, stream);
```

只有所有 `nranks` 个等待条件都满足，当前 rank 才越过屏障。


## 9. RMA All-to-all

每个 rank 针对每个 peer 调用 `ncclPutSignal`：将发往 peer 的数据块写入其 window 的指定偏移，并随后等待来自所有 rank 的信号。多个远程写操作应放在 group 中。

```cpp
// 先确保所有 window 和源缓冲区已就绪；可用 Barrier 建立此前提。
ncclGroupStart();
for (int peer = 0; peer < nranks; ++peer) {
  size_t offset = peer * count * wordSize(datatype);
  ncclPutSignal(sendbuf[peer], count, datatype, peer, window, offset,
                 /* signal 参数 */, comm, stream);
}
ncclGroupEnd();
ncclWaitSignal(nranks, waitDescs, comm, stream);
```

调用 `PutSignal` 前必须确保对端 window 和缓冲区准备完成，否则会产生无效访问或数据竞争。


## 10. 检查清单

1. 每条 `ncclSend` 都有唯一匹配的 `ncclRecv`，且 peer、count、datatype 正确。
2. 将相互依赖且需并行推进的不同 peer 操作合入同一 group。
3. 同一 peer 的多条消息在两端保持同样顺序。
4. RMA 前注册 window，并在 PutSignal 前保证来源和远端目标均已就绪。
5. 使用结果、注销 window 或释放内存前，正确处理 CUDA stream 的完成依赖。

参考：[NCCL P2P Usage](https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/usage/p2p.html)；[P2P API](https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/api/p2p.html)。
